# AMS-SkipGNN Kaggle T4 runner

Use **GPU T4** and **Save & Commit All**. This notebook clones `aryonmt/finalProject`, fetches SkipGNN fold-1 splits, runs smoke tests, then Stage-1 training. Printed logs are the archived outputs — after Kaggle finishes, commit this notebook back into the repo.


In [ ]:
import os, sys, platform, subprocess, shutil
from pathlib import Path

print('python', sys.version)
print('platform', platform.platform())
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu', torch.cuda.get_device_name(0))
except Exception as e:
    print('torch import failed', e)

REPO = 'https://github.com/aryonmt/finalProject.git'
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
ROOT = WORK / 'finalProject'
if (Path.cwd() / 'src' / 'models').exists():
    ROOT = Path.cwd()
    print('already in repo', ROOT)
else:
    if ROOT.exists():
        shutil.rmtree(ROOT)
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO, str(ROOT)])
    print('cloned', ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('cwd', os.getcwd())


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'])
print('pip install -e . done')
subprocess.check_call([sys.executable, 'scripts/fetch_data.py'])
print('data fetch done')


In [ ]:
import subprocess, sys
rc = subprocess.call([sys.executable, '-m', 'pytest', '-q', 'tests/test_smoke.py'])
print('pytest rc', rc)
assert rc == 0, 'smoke tests failed'


In [ ]:
import os, subprocess, sys, time
stage = os.environ.get('STAGE', '1')
print('STAGE', stage)
t0 = time.time()
cmd = [sys.executable, 'scripts/run_benchmark.py', '--dataset', 'DTI', '--models', 'gcn', 'skipgnn', 'ams', '--device', 'auto']
if stage == '1':
    cmd += ['--quick']
print('running', cmd)
subprocess.check_call(cmd)
print('DTI minutes', round((time.time()-t0)/60, 2))
if stage != '1':
    subprocess.check_call([sys.executable, 'scripts/run_benchmark.py', '--dataset', 'DDI', '--models', 'gcn', 'skipgnn', 'ams'])
print('benchmark done')


In [ ]:
import os, subprocess, sys
stage = os.environ.get('STAGE', '1')
if stage == '1':
    print('skipping ablation/robustness in STAGE=1; set STAGE=2 for full suite')
else:
    subprocess.check_call([sys.executable, 'scripts/run_ablation.py', '--dataset', 'DTI'])
    subprocess.check_call([sys.executable, 'scripts/run_robustness.py', '--dataset', 'DTI'])
    subprocess.check_call([sys.executable, 'scripts/run_benchmark.py', '--dataset', 'PPI', '--models', 'skipgnn', 'ams'])
    subprocess.check_call([sys.executable, 'scripts/run_benchmark.py', '--dataset', 'GDI', '--models', 'skipgnn', 'ams'])
print('stage extras done')


In [ ]:
import json, subprocess, sys
from pathlib import Path
subprocess.call([sys.executable, 'scripts/make_figures.py'])
for p in sorted(Path('results').rglob('*.json')):
    print('===', p, '===')
    print(p.read_text(encoding='utf-8')[:4000])
print('figures', list(Path('figures').glob('*.png')))
print('KAGGLE RUN COMPLETE')
